# 🔍 02 - Patch Extraction & Augmentation Visualization

This notebook demonstrates:
1. **256×256 Patch Tiling** with configurable overlap (64px train overlap, 0px validation/test tiling).
2. **Patch Manifest Inspection** (identifying patches containing positive oil spill slicks).
3. **Albumentations Augmentation Pipeline** (verifying label preservation under flips, rotations, and intensity shifts).

In [ ]:
import sys
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add src to path
sys.path.append(str(Path.cwd().parent / "src"))

from oilspill.data.dataset import mask_to_rgb
from oilspill.data.transforms import get_training_transforms

## 1. Inspect Patch Manifest

In [ ]:
manifest_path = Path("../data/splits/patch_manifest.csv")
if manifest_path.exists():
    df = pd.read_csv(manifest_path)
    print(f"Total Patches: {len(df):,}")
    print(f"Patches with Oil: {df['has_oil'].sum():,} ({df['has_oil'].mean()*100:.2f}%)")
    display(df.head(10))
else:
    print("Manifest not found. Run 'python scripts/prepare_data.py' first.")

## 2. Visualize Patches Containing Oil Spills vs Look-alikes

In [ ]:
if manifest_path.exists():
    oil_patches = df[df["has_oil"] == True]
    if len(oil_patches) > 0:
        sample_rows = oil_patches.sample(min(4, len(oil_patches)), random_state=42)
        
        fig, axes = plt.subplots(len(sample_rows), 2, figsize=(8, 4 * len(sample_rows)))
        if len(sample_rows) == 1:
            axes = np.expand_dims(axes, 0)
            
        for i, (_, row) in enumerate(sample_rows.iterrows()):
            img = cv2.imread(str(Path("..") / row["image_path"]), cv2.IMREAD_GRAYSCALE)
            msk = cv2.imread(str(Path("..") / row["mask_path"]), cv2.IMREAD_UNCHANGED)
            
            axes[i, 0].imshow(img, cmap="gray")
            axes[i, 0].set_title(f"SAR Patch: {row['patch_id']}")
            axes[i, 0].axis("off")
            
            axes[i, 1].imshow(mask_to_rgb(msk))
            axes[i, 1].set_title(f"Oil Pixels: {row['oil_pixels']:,}")
            axes[i, 1].axis("off")
            
        plt.tight_layout()
        plt.show()